<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 09 — Random Forest e Ensembles
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Centenas de Árvores Votando no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🌲🌲🌲 6º Modelo do Curso</span>
</div>


## Onde estamos?

| Aula | Modelo | Ideia central |
|------|--------|---------------|
| 04 | Reg. Linear | Prever um número via reta |
| 05 | Reg. Logística | Probabilidade via sigmoide |
| 06 | KNN | Votação dos K vizinhos mais próximos |
| 07 | SVM | Hiperplano de margem máxima |
| 08 | Árvore de Decisão | Perguntas encadeadas sobre as features |
| **09** | **Random Forest** | **Centenas de árvores, cada uma vê um pedaço diferente dos dados — e elas votam** |

Na Aula 08 você viu que uma única Árvore de Decisão tem um problema: se
deixada crescer livremente, ela **decora** o treino (overfitting). Mesmo
limitando a profundidade, uma única árvore ainda é **instável** — trocar
alguns poucos passageiros no treino pode mudar completamente as perguntas que
ela escolhe.

E se, em vez de confiar em uma única árvore, treinássemos **centenas delas**,
cada uma vendo um pedaço ligeiramente diferente dos dados, e deixássemos todas
**votarem** na previsão final?

> *"Uma árvore pode errar sozinha. Centenas de árvores, cada uma vendo os*
> *dados de um jeito um pouco diferente, raramente erram da mesma forma —*
> *e o erro de uma é compensado pelo acerto das outras."*

Essa é a ideia do **Random Forest**: um exemplo de **ensemble** (conjunto de
modelos) baseado em **bagging** (Bootstrap Aggregating).

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Por que combinar modelos? — a sabedoria das multidões |
| **2** | Bootstrap e Feature Randomness — como cada árvore fica diferente |
| **3** | Treinando o Random Forest no Titanic |
| **4** | Quantas árvores? — `n_estimators` e overfitting |
| **5** | Feature Importance e placar final |

> **Tempo estimado: 35 minutos**

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#155724;">Por que essa aula importa tanto:</strong>
<span style="color:#155724;"> Random Forest é um dos modelos mais usados em produção com dados tabulares — robusto, difícil de "quebrar" com hiperparâmetros errados, e quase sempre melhor que uma árvore única. É também a porta de entrada para o Gradient Boosting (XGBoost, LightGBM), que você pode encontrar depois deste curso.</span>
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# As mesmas 5 features usadas desde a Aula 05
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")
print()
print("Nota: assim como a Árvore de Decisão (Aula 08), o Random Forest")
print("NÃO precisa de normalização — ele também usa pontos de corte.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Por Que Combinar Modelos? — A Sabedoria das Multidões</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Um especialista pode errar. Uma multidão de especialistas independentes, raramente."</p>
    </div>
</div>

### O problema da árvore única

Na Aula 08, vimos que uma Árvore de Decisão é fácil de interpretar, mas tem
um problema: ela é **instável**. Mude alguns poucos passageiros no conjunto
de treino, e a árvore pode escolher perguntas completamente diferentes desde
a raiz. Isso é chamado de **alta variância**.

### A ideia do ensemble

Em vez de confiar em uma única árvore (um "especialista"), o Random Forest
treina **centenas de árvores diferentes** e combina as previsões de todas —
por **votação** (a classe mais votada ganha).

```
Árvore 1 → prevê "Sobreviveu"
Árvore 2 → prevê "Não Sobreviveu"
Árvore 3 → prevê "Sobreviveu"
   ⋮
Árvore 200 → prevê "Sobreviveu"
─────────────────────────────────
Previsão final = classe mais votada entre as 200 árvores
```

Essa técnica se chama **bagging** (Bootstrap Aggregating). A sacada é: se
cada árvore erra de um jeito **diferente** e **independente**, os erros se
cancelam na votação, e o resultado combinado é mais estável e preciso do que
qualquer árvore individual.

<div style="background:#e3f2fd; border-left:5px solid #1565c0; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#1565c0;">Analogia:</strong>
<span style="color:#1565c0;"> pense num júri de 200 pessoas em vez de um único juiz. Se cada jurado vê o caso de um ângulo um pouco diferente, o veredito da maioria tende a ser mais justo do que a opinião de uma pessoa só — mesmo que alguns jurados individualmente estejam errados.</span>
</div>

Mas isso só funciona se as árvores forem **realmente diferentes** entre si.
Se todas vissem exatamente os mesmos dados, todas chegariam à mesma árvore —
e votar entre cópias idênticas não ajuda em nada. É aí que entra a Parte 2.


In [ ]:
# Demonstração simples: um "júri" de pequenas árvores vota melhor que uma só?
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split as tts

# Usando um recorte pequeno e ruidoso de propósito, para expor a instabilidade
X_pequeno, _, y_pequeno, _ = tts(X_treino, y_treino, train_size=80, random_state=1, stratify=y_treino)

# Treinando 5 árvores pequenas, cada uma vendo uma AMOSTRA diferente dos 80 exemplos
np.random.seed(42)
previsoes_arvores = []
for i in range(5):
    idx = np.random.choice(len(X_pequeno), len(X_pequeno), replace=True)  # amostra com reposição
    arv = DecisionTreeClassifier(max_depth=3, random_state=i)
    arv.fit(X_pequeno.values[idx], y_pequeno.values[idx])
    previsoes_arvores.append(arv.predict(X_teste.values))

previsoes_arvores = np.array(previsoes_arvores)

# Votação: a classe mais comum entre as 5 árvores
from scipy.stats import mode
votacao = mode(previsoes_arvores, axis=0, keepdims=False).mode

acc_individuais = [(p == y_teste.values).mean() for p in previsoes_arvores]
acc_votacao      = (votacao == y_teste.values).mean()

print("Acurácia de cada árvore individual (treinada com poucos dados, instável):")
for i, acc in enumerate(acc_individuais, 1):
    print(f"  Árvore {i}: {acc:.1%}")

print()
print(f"Acurácia da VOTAÇÃO das 5 árvores juntas: {acc_votacao:.1%}")
print()
print("Mesmo com árvores individuais instáveis, a votação tende a ser")
print("mais estável e, em geral, melhor que a média das árvores sozinhas.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Olhando os resultados acima: (a) as 5 árvores individuais têm a mesma acurácia? Por que não, se foram treinadas com o "mesmo" conjunto de 80 passageiros? (b) a votação das 5 árvores foi melhor, igual ou pior que a média das árvores individuais?</span></div>

*✏️ (a) As árvores têm acurácias diferentes porque: `???`*

*✏️ (b) A votação foi `???` em relação à média das árvores individuais*


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("(a) Cada árvore viu uma AMOSTRA diferente dos mesmos 80 passageiros")
# print("    (reamostragem com reposição — alguns passageiros aparecem repetidos,")
# print("    outros nem aparecem). Isso faz cada árvore aprender perguntas")
# print("    ligeiramente diferentes, e por isso ter acurácia diferente.")
# print()
# print("(b) A votação combinada costuma empatar ou superar a média das árvores")
# print("    individuais, porque os erros de cada árvore tendem a não se repetir")
# print("    nos mesmos passageiros — a maioria 'corrige' o erro da minoria.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Bootstrap e Feature Randomness — Como Cada Árvore Fica Diferente</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Para a votação funcionar, as árvores precisam discordar de forma útil."</p>
    </div>
</div>

O Random Forest usa **duas fontes de aleatoriedade** para garantir que cada
árvore seja diferente das outras — sem isso, todas as árvores ficariam
idênticas e a votação não ajudaria em nada.

### 1. Bootstrap — cada árvore vê uma amostra diferente das LINHAS

Cada árvore é treinada com uma **amostra aleatória com reposição** do
conjunto de treino original (do mesmo tamanho, mas sorteada com repetição —
foi isso que fizemos manualmente na Parte 1). Alguns passageiros aparecem
repetidos nessa amostra, outros não aparecem — em média, cada árvore vê
cerca de 63% dos passageiros originais, alguns deles mais de uma vez.

### 2. Feature Randomness — cada divisão vê um subconjunto das COLUNAS

Além disso, em cada divisão de cada árvore, o algoritmo **não testa todas as
features** — ele sorteia um subconjunto delas e escolhe a melhor divisão
apenas entre as sorteadas. Isso evita que todas as árvores "convirjam" sempre
para a mesma pergunta na raiz (ex: sempre `sex_enc` primeiro), forçando
diversidade entre elas.

```
Árvore 1: viu passageiros {A, A, B, D, D, E, ...} | testou {pclass, idade} em cada nó
Árvore 2: viu passageiros {B, C, C, D, F, F, ...} | testou {sexo, tarifa} em cada nó
Árvore 3: viu passageiros {A, B, B, C, E, E, ...} | testou {idade, família} em cada nó
   ⋮
```

Juntas, essas duas fontes de aleatoriedade fazem o Random Forest **reduzir a
variância** (instabilidade) da árvore única, sem perder muito em viés.


In [ ]:
# Visualizando o bootstrap: quantos passageiros únicos sobram em cada amostra?
np.random.seed(42)
n = len(X_treino)

for i in range(3):
    idx_bootstrap = np.random.choice(n, n, replace=True)
    unicos = len(set(idx_bootstrap))
    print(f"Amostra bootstrap {i+1}: {unicos} passageiros únicos de {n} "
          f"({unicos/n:.0%}) — o resto são repetições")

print()
print("Esse ~63% é uma propriedade matemática do bootstrap: em média, cada")
print("amostra deixa de fora cerca de 1/3 dos dados originais.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Pense e responda: (a) por que amostrar SEM reposição (cada árvore vendo um pedaço disjunto dos dados) seria pior que amostrar COM reposição (bootstrap)? (b) por que sortear um subconjunto de features em cada divisão ajuda a diversificar as árvores?</span></div>

*✏️ (a) Amostrar sem reposição seria pior porque: `???`*

*✏️ (b) Sortear features em cada divisão ajuda porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("(a) Sem reposição, cada árvore veria só uma fração pequena e fixa dos")
# print("    dados (ex: 1/10 se quiséssemos 10 árvores), perdendo muita informação.")
# print("    Com reposição (bootstrap), cada árvore pode usar um conjunto do MESMO")
# print("    tamanho do treino original, só que reamostrado — aproveitando melhor")
# print("    os dados disponíveis.")
# print()
# print("(b) Sem sortear features, a árvore quase sempre escolheria a mesma")
# print("    feature mais forte (ex: sex_enc) como primeira pergunta em TODAS as")
# print("    árvores, tornando-as muito parecidas entre si. Sortear features força")
# print("    cada árvore a explorar caminhos diferentes, mesmo que às vezes menos")
# print("    'ótimos' individualmente — o que aumenta a diversidade do conjunto.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Treinando o Random Forest no Titanic</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A mesma interface de sempre — fit() e predict() — só que por dentro são centenas de árvores."</p>
    </div>
</div>

Vamos treinar o Random Forest e comparar diretamente com a Árvore de Decisão
única que treinamos na Aula 08, nos mesmos dados.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ── Árvore única (igual à Aula 08, para comparação) ──────────────────────────
arvore_unica = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42)
arvore_unica.fit(X_treino, y_treino)

# ── Random Forest: 200 árvores votando juntas ────────────────────────────────
floresta = RandomForestClassifier(n_estimators=200, max_depth=5,
                                   min_samples_leaf=5, random_state=42)
floresta.fit(X_treino, y_treino)

y_pred_arvore    = arvore_unica.predict(X_teste)
y_pred_floresta  = floresta.predict(X_teste)

print("COMPARAÇÃO — Árvore Única vs Random Forest (200 árvores)")
print(f"  {'Modelo':<20} {'Acurácia':>10} {'F1-Score':>10}")
print(f"  {'Árvore Única':<20} {accuracy_score(y_teste, y_pred_arvore):>10.4f} "
      f"{f1_score(y_teste, y_pred_arvore):>10.4f}")
print(f"  {'Random Forest':<20} {accuracy_score(y_teste, y_pred_floresta):>10.4f} "
      f"{f1_score(y_teste, y_pred_floresta):>10.4f}")


In [ ]:
# Estabilidade: treinando 30 vezes com pequenas variações no treino
# (sorteamos 90% dos dados de treino a cada rodada, para simular dados "um pouco diferentes")
print("Testando ESTABILIDADE — 30 rodadas, cada uma com 90% do treino sorteado")

accs_arvore, accs_floresta = [], []
for seed in range(30):
    idx = np.random.RandomState(seed).choice(len(X_treino), int(len(X_treino)*0.9), replace=False)
    X_sub, y_sub = X_treino.values[idx], y_treino.values[idx]

    arv = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42)
    arv.fit(X_sub, y_sub)
    accs_arvore.append(arv.score(X_teste, y_teste))

    rf = RandomForestClassifier(n_estimators=200, max_depth=5,
                                 min_samples_leaf=5, random_state=42)
    rf.fit(X_sub, y_sub)
    accs_floresta.append(rf.score(X_teste, y_teste))

print(f"  Acurácia média  — Árvore: {np.mean(accs_arvore):.1%}  |  Floresta: {np.mean(accs_floresta):.1%}")
print(f"  Desvio padrão   — Árvore: {np.std(accs_arvore):.4f}  |  Floresta: {np.std(accs_floresta):.4f}")
print()
print("Quanto menor o desvio padrão, mais ESTÁVEL é o modelo —")
print("menos sensível a qual parte dos dados foi usada no treino.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Olhando a tabela de estabilidade: (a) qual modelo tem desvio padrão menor? (b) o que isso significa na prática, se você fosse colocar o modelo em produção e treiná-lo novamente no mês seguinte com dados ligeiramente diferentes?</span></div>

*✏️ (a) Modelo com desvio padrão menor: `???`*

*✏️ (b) Na prática, isso significa: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("(a) O Random Forest costuma ter desvio padrão bem menor que a árvore única.")
# print()
# print("(b) Um modelo mais estável significa que, se você retreinar com dados")
# print("    ligeiramente diferentes (mais passageiros, uma nova amostra), o")
# print("    desempenho esperado continua parecido. Uma árvore única pode variar")
# print("    muito de uma versão do dataset para outra — o que é arriscado em produção,")
# print("    onde você quer previsibilidade no comportamento do modelo ao longo do tempo.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Quantas Árvores? — n_estimators e Overfitting</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Mais árvores quase sempre ajudam — mas até um certo ponto, e com um custo."</p>
    </div>
</div>

O hiperparâmetro mais importante do Random Forest é `n_estimators` — o
**número de árvores** na floresta.

```
Poucas árvores (ex: 5)   → votação instável, parecido com usar só 1 árvore
Muitas árvores (ex: 500) → votação mais estável, mas demora mais para treinar
```

**Boa notícia:** diferente da Árvore de Decisão (onde profundidade alta
sempre arrisca overfitting), adicionar MAIS árvores ao Random Forest quase
nunca piora o desempenho no teste — o ganho só desacelera e estabiliza.
O "custo" de usar árvores demais é só **tempo de treino e de predição**, não
overfitting.


In [ ]:
# Testando diferentes números de árvores
valores_n = [1, 5, 10, 25, 50, 100, 200, 400]
accs_teste  = []

for n in valores_n:
    rf = RandomForestClassifier(n_estimators=n, max_depth=5,
                                 min_samples_leaf=5, random_state=42)
    rf.fit(X_treino, y_treino)
    accs_teste.append(rf.score(X_teste, y_teste))

plt.figure(figsize=(9, 5))
plt.plot(valores_n, [a*100 for a in accs_teste], "o-", color="#0f3460", linewidth=2.5, markersize=8)
plt.xlabel("Número de Árvores (n_estimators)")
plt.ylabel("Acurácia no Teste (%)")
plt.title("Random Forest — Acurácia vs Número de Árvores", fontweight="bold")
plt.xscale("log")
plt.tight_layout()
plt.show()

for n, acc in zip(valores_n, accs_teste):
    print(f"  n_estimators={n:>4}: acurácia = {acc:.1%}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Olhando o gráfico: (a) a partir de quantas árvores o ganho de acurácia fica pequeno (a curva "estabiliza")? (b) por que não escolher sempre o maior n_estimators possível, já que ele raramente piora a acurácia?</span></div>

*✏️ (a) A curva estabiliza por volta de n_estimators = `???`*

*✏️ (b) Não escolhemos sempre o maior n_estimators porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("(a) Geralmente entre 50 e 200 árvores a curva já estabilizou — adicionar")
# print("    mais não traz ganho relevante de acurácia.")
# print()
# print("(b) Mais árvores significam mais tempo de treino e, principalmente, mais")
# print("    tempo de PREDIÇÃO (cada árvore precisa ser consultada a cada previsão).")
# print("    Em produção, com milhões de previsões por dia, isso tem custo real de")
# print("    infraestrutura — sem ganho de qualidade proporcional além de certo ponto.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Feature Importance e Placar Final</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Centenas de opiniões sobre quais features importam, tiradas na média."</p>
    </div>
</div>

Assim como a árvore única, o Random Forest também calcula **Feature
Importance** — mas agora é a **média da importância de cada feature ao longo
de todas as árvores da floresta**, o que tende a ser mais estável e confiável
do que a importância calculada por uma única árvore.


In [ ]:
# Feature importance do Random Forest vs da árvore única
imp_floresta = pd.DataFrame({
    "feature":    FEATURES,
    "Random Forest": floresta.feature_importances_,
    "Árvore Única":  arvore_unica.feature_importances_,
}).sort_values("Random Forest", ascending=False)

x = np.arange(len(FEATURES))
largura = 0.35

plt.figure(figsize=(8, 5))
plt.barh(x - largura/2, imp_floresta["Random Forest"], height=largura,
         color="#0f3460", label="Random Forest")
plt.barh(x + largura/2, imp_floresta["Árvore Única"], height=largura,
         color="#e94560", label="Árvore Única")
plt.yticks(x, imp_floresta["feature"])
plt.xlabel("Importância")
plt.title("Feature Importance — Random Forest vs Árvore Única", fontweight="bold")
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(imp_floresta.round(4).to_string(index=False))


### Comparando com os modelos anteriores

Vamos treinar rapidamente o KNN e a Regressão Logística (Aulas 05 e 06) para
fechar com um placar entre os quatro modelos.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score, roc_curve

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

knn_ref = KNeighborsClassifier(n_neighbors=7).fit(X_treino_sc, y_treino)
lr_ref  = LogisticRegression(max_iter=1000, random_state=42).fit(X_treino_sc, y_treino)

modelos = {
    "KNN (K=7)":           (knn_ref,       X_teste_sc),
    "Reg. Logística":      (lr_ref,        X_teste_sc),
    "Árvore Única":        (arvore_unica,  X_teste),
    "Random Forest":       (floresta,      X_teste),
}

tabela = []
for nome, (modelo, Xte) in modelos.items():
    yp  = modelo.predict(Xte)
    ypr = modelo.predict_proba(Xte)[:, 1]
    tabela.append({
        "Modelo":   nome,
        "Acurácia": accuracy_score(y_teste, yp),
        "Precisão": precision_score(y_teste, yp),
        "Recall":   recall_score(y_teste, yp),
        "F1":       f1_score(y_teste, yp),
        "AUC-ROC":  roc_auc_score(y_teste, ypr),
    })

df_tabela = pd.DataFrame(tabela).set_index("Modelo")
print("PLACAR FINAL — KNN vs Reg. Logística vs Árvore vs Random Forest")
print(df_tabela.round(4).to_string())


In [ ]:
# Curvas ROC sobrepostas
plt.figure(figsize=(7, 6))
cores = ["#a8d8ea", "#0f3460", "#f0a500", "#e94560"]
for cor, (nome, (modelo, Xte)) in zip(cores, modelos.items()):
    ypr = modelo.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_teste, ypr)
    auc = roc_auc_score(y_teste, ypr)
    lw = 3 if nome == "Random Forest" else 1.8
    plt.plot(fpr, tpr, color=cor, linewidth=lw, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "k--", linewidth=1, alpha=0.4, label="Aleatório")
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo (Recall)")
plt.title("Curvas ROC — Comparação Final", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final) — Analisando a tabela e as curvas ROC: (a) o Random Forest superou a Árvore Única? Em qual métrica a diferença é mais clara? (b) se você precisasse explicar uma decisão específica para um passageiro ("por que meu modelo previu isso?"), seria mais fácil com a Árvore Única ou o Random Forest? Por quê?</span></div>

*✏️ (a) O Random Forest superou a Árvore em: `???`*

*✏️ (b) Mais fácil de explicar uma decisão individual: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print(f"(a) Comparando diretamente com a tabela acima:")
# for col in ["Acurácia", "Precisão", "Recall", "F1", "AUC-ROC"]:
#     diff = df_tabela.loc["Random Forest", col] - df_tabela.loc["Árvore Única", col]
#     print(f"    {col:<10}: {'+' if diff >= 0 else ''}{diff:.4f}")
# print()
# print("    O Random Forest costuma ganhar em AUC-ROC (qualidade geral das")
# print("    probabilidades, olhando todos os limiares possíveis) — mesmo quando")
# print("    uma métrica isolada como Recall ou Acurácia no limiar padrão (0.5)")
# print("    não mostra vantagem clara. Isso é esperado: o ensemble produz")
# print("    probabilidades mais bem calibradas, não necessariamente uma classe")
# print("    final diferente para cada passageiro.")
# print()
# print("(b) A Árvore Única é mais fácil de explicar individualmente: você pode")
# print("    mostrar o caminho exato de 3-4 perguntas que levaram à previsão.")
# print("    No Random Forest, a previsão vem da VOTAÇÃO de 200 árvores diferentes —")
# print("    não existe um único caminho de perguntas para mostrar. Você ganha em")
# print("    precisão e estabilidade, mas perde a explicabilidade direta 'nó a nó'.")
# print("    Essa é a troca clássica: interpretabilidade vs performance.")


**✏️ Minha reflexão sobre a aula:**

1. Random Forest em uma frase simples: *...*

2. A diferença entre bagging e uma árvore única é: *...*

3. Eu preferiria uma Árvore Única em vez de um Random Forest quando: *...*
